# Lab12 Preset Path Execution

This notebook executes a fixed offboard waypoint path. During the main run, Python computes each segment turn angle and drive distance, then sends `TURN_REL_DEG` and `DRIVE_CELL_MM` commands over BLE.

Place the robot at `(-4, -3)` facing about `45 deg` toward `(-2, -1)` before running the preset path.


In [52]:
%load_ext autoreload
%autoreload 2

import os
import time
import json
import math
from pathlib import Path

# Keep the same working-directory convention as the earlier BLE notebooks.
if not Path("connection.yaml").exists() and Path("ble_python/connection.yaml").exists():
    os.chdir("ble_python")

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
from lab12_local_planning import (
    Lab12GridPlanner,
    Lab12Localizer,
    point_in_polygon,
    point_segment_distance,
    wrap_deg,
)

LOG.propagate = False

ble = get_ble_controller()
ble.connect()
print("BLE connected")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-05-07 15:43:32,092 | INFO     |: Already connected to a BLE device
BLE connected


## Notify Handler

Starts BLE notifications so notebook cells can wait for `TURN_DONE`, `DRIVE_DONE`, `DRIVE_STOPPED_TOF`, and `NAV` log messages.


In [33]:
rx_msgs = []
map_samples_mm = {}
nav_rows = []

def lab12_handler(uuid, byte_array):
    msg = byte_array.decode(errors="replace").strip("\x00\r\n ")
    if not msg:
        return

    rx_msgs.append(msg)
    print(msg)

    if msg.startswith("MAP,"):
        parts = msg.split(",")
        if len(parts) >= 5:
            map_samples_mm[int(parts[1])] = float(parts[4])
    elif msg.startswith("MAP_SAMPLE,"):
        parts = msg.split(",")
        if len(parts) >= 4:
            map_samples_mm[int(parts[1])] = float(parts[3])
    elif msg.startswith("NAV,"):
        nav_rows.append(msg.split(","))

def wait_for_prefix(prefixes, timeout_s=10.0, start_index=None):
    if isinstance(prefixes, str):
        prefixes = (prefixes,)
    if start_index is None:
        start_index = len(rx_msgs)

    deadline = time.time() + timeout_s
    checked = start_index
    while time.time() < deadline:
        while checked < len(rx_msgs):
            msg = rx_msgs[checked]
            checked += 1
            if any(msg.startswith(p) for p in prefixes):
                return msg
        ble.sleep(0.05)
    raise TimeoutError(f"Timed out waiting for {prefixes}")

try:
    ble.stop_notify(ble.uuid["RX_STRING"])
except Exception:
    pass

ble.start_notify(ble.uuid["RX_STRING"], lab12_handler)
print("Notify started")


Notify started


## Preset Path Preview

This cell defines the fixed feet-coordinate waypoint list and prints the turn angle, drive distance, estimated drive duration, and wall clearance for each segment. It does not move the robot.


In [34]:
PRESET_PATH_FT = [
    (-4, -3),  # start
    (-2, -1),
    (1, -1),
    (2, -3),
    (5, -3),
    (5, -2),
    (5, 3),
    (0, 3),
    (0, 0),   # end
]

START_CELL = PRESET_PATH_FT[0]
END_CELL = PRESET_PATH_FT[-1]
START_HEADING_DEG = 45.0  # place the robot at (-4,-3), pointing along the first segment
WALL_INFLATION_FT = 0.5
MS_PER_FT = 900
BASE_PWM = 90
FRONT_STOP_MM = 250

planner = Lab12GridPlanner(wall_inflation_ft=WALL_INFLATION_FT)

def segment_clearance_ft(segment_start, segment_goal, samples=100):
    min_clearance = float("inf")
    outside_outer_wall = False

    for i in range(samples + 1):
        t = i / samples
        x = segment_start[0] + (segment_goal[0] - segment_start[0]) * t
        y = segment_start[1] + (segment_goal[1] - segment_start[1]) * t

        if not point_in_polygon((x, y), planner.outer_polygon_ft):
            outside_outer_wall = True

        for wall in planner.internal_segments_ft:
            min_clearance = min(min_clearance, point_segment_distance((x, y), wall))

    return min_clearance, outside_outer_wall

def preset_segments(path=PRESET_PATH_FT, start_heading_deg=START_HEADING_DEG, ms_per_ft=MS_PER_FT):
    current_heading = start_heading_deg
    segments = []

    for idx, (segment_start, segment_goal) in enumerate(zip(path, path[1:]), start=1):
        dx = segment_goal[0] - segment_start[0]
        dy = segment_goal[1] - segment_start[1]
        distance_ft = math.hypot(dx, dy)
        desired_heading = wrap_deg(math.degrees(math.atan2(dy, dx)))
        turn_delta = wrap_deg(desired_heading - current_heading)
        min_clearance, outside_outer_wall = segment_clearance_ft(segment_start, segment_goal)

        segments.append({
            "segment": idx,
            "start": segment_start,
            "goal": segment_goal,
            "dx": dx,
            "dy": dy,
            "distance_ft": distance_ft,
            "distance_mm": distance_ft * 304.8,
            "desired_heading_deg": desired_heading,
            "turn_delta_deg": turn_delta,
            "drive_duration_ms": int(round(distance_ft * ms_per_ft)),
            "min_internal_wall_clearance_ft": min_clearance,
            "outside_outer_wall": outside_outer_wall,
        })
        current_heading = desired_heading

    return segments

for point in PRESET_PATH_FT:
    if not planner.is_free(point):
        raise ValueError(f"Preset path point is blocked or outside the map: {point}")

print("Preset path points:", PRESET_PATH_FT)
print(planner.render_ascii(route=PRESET_PATH_FT, start=START_CELL, goal=END_CELL))
for seg in preset_segments():
    print(
        f"Segment {seg['segment']}: {seg['start']} -> {seg['goal']} | "
        f"turn={seg['turn_delta_deg']:.1f} deg, heading={seg['desired_heading_deg']:.1f} deg, "
        f"drive={seg['distance_ft']:.3f} ft ({seg['distance_mm']:.0f} mm), "
        f"duration={seg['drive_duration_ms']} ms, clearance={seg['min_internal_wall_clearance_ft']:.2f} ft"
    )


Preset path points: [(-4, -3), (-2, -1), (1, -1), (2, -3), (5, -3), (5, -2), (5, 3), (0, 3), (0, 0)]
     -5 -4 -3 -2 -1  0  1  2  3  4  5  6
y= 4  #  #  #  .  .  .  .  .  .  .  .  .
y= 3  #  #  #  .  .  *  .  .  .  .  *  .
y= 2  #  #  #  .  .  .  .  .  #  #  .  .
y= 1  #  #  #  .  .  .  .  #  #  #  #  .
y= 0  .  .  .  .  .  G  .  #  #  #  #  .
y=-1  .  .  .  *  .  .  *  .  #  #  .  .
y=-2  .  .  .  .  .  #  .  .  .  .  *  .
y=-3  .  S  .  .  #  #  #  *  .  .  *  .
y=-4  .  .  .  .  #  #  #  .  .  .  .  .
Segment 1: (-4, -3) -> (-2, -1) | turn=0.0 deg, heading=45.0 deg, drive=2.828 ft (862 mm), duration=2546 ms, clearance=2.12 ft
Segment 2: (-2, -1) -> (1, -1) | turn=-45.0 deg, heading=0.0 deg, drive=3.000 ft (914 mm), duration=2700 ms, clearance=1.50 ft
Segment 3: (1, -1) -> (2, -3) | turn=-63.4 deg, heading=-63.4 deg, drive=2.236 ft (682 mm), duration=2012 ms, clearance=1.12 ft
Segment 4: (2, -3) -> (5, -3) | turn=63.4 deg, heading=0.0 deg, drive=3.000 ft (914 mm), duration=2700 ms, 

## Motion Primitives

Low-level BLE helpers for relative turns, timed straight drives, emergency stop, and motion-log download. Running the definition cell alone does not move the robot.


In [35]:
def send_turn(delta_deg, timeout_ms=3500, kp=0.8, ki=0.001, kd=0.2):
    start_idx = len(rx_msgs)
    cmd_str = f"{delta_deg}|{timeout_ms}|{kp}|{ki}|{kd}"
    ble.send_command(CMD.TURN_REL_DEG, cmd_str)
    return wait_for_prefix(("TURN_DONE", "TURN_FAILED"),
                           timeout_s=timeout_ms / 1000.0 + 1.5,
                           start_index=start_idx)

def drive_cell(distance_mm=304.8, base_pwm=90, duration_ms=900, heading_kp=1.2, front_stop_mm=250):
    start_idx = len(rx_msgs)
    cmd_str = f"{distance_mm}|{base_pwm}|{duration_ms}|{heading_kp}|{front_stop_mm}"
    ble.send_command(CMD.DRIVE_CELL_MM, cmd_str)
    return wait_for_prefix(("DRIVE_DONE", "DRIVE_FAILED", "DRIVE_STOPPED_TOF"),
                           timeout_s=duration_ms / 1000.0 + 2.0,
                           start_index=start_idx)

def stop_nav():
    ble.send_command(CMD.STOP_NAV, "")

def request_nav_log(timeout_s=10.0):
    nav_rows.clear()
    start_idx = len(rx_msgs)
    ble.send_command(CMD.SEND_NAV_LOG, "")
    wait_for_prefix("NAV_END", timeout_s=timeout_s, start_index=start_idx)
    return list(nav_rows)


In [36]:
# Primitive test examples. Run these one at a time when the car is lifted or in a safe area.
# send_turn(90)
# send_turn(-90)
# send_turn(180, timeout_ms=5000)
# drive_cell(base_pwm=90, duration_ms=900, front_stop_mm=250)
# stop_nav()


## Optional Scan/Localization Tools

These helpers are kept for debugging only. The main preset-path run below does not call scan/localization automatically.


In [37]:
localizer = Lab12Localizer(planner, sensor_sigma_m=0.18)

def run_map_scan(step_deg=20.0,
                 num_steps=18,
                 kp=0.8,
                 ki=0.001,
                 kd=0.5,
                 settle_ms=550,
                 timeout_s=35.0):
    map_samples_mm.clear()
    start_idx = len(rx_msgs)
    cmd_str = f"{step_deg}|{num_steps}|{kp}|{ki}|{kd}|{settle_ms}"
    ble.send_command(CMD.START_MAP_SCAN, cmd_str)
    status = wait_for_prefix(("MAP_SCAN_DONE", "MAP_SCAN_YAW_INVALID", "MAP_SCAN_BAD_ARGS"),
                             timeout_s=timeout_s,
                             start_index=start_idx)
    if not status.startswith("MAP_SCAN_DONE"):
        raise RuntimeError(status)

    map_samples_mm.clear()
    start_idx = len(rx_msgs)
    ble.send_command(CMD.SEND_MAP_SCAN, "")
    wait_for_prefix("MAP_END", timeout_s=10.0, start_index=start_idx)

    if len(map_samples_mm) < num_steps:
        raise RuntimeError(f"Expected {num_steps} map samples, got {len(map_samples_mm)}")

    samples_cw_mm = [map_samples_mm[i] for i in range(num_steps)]
    samples_ccw_mm = [samples_cw_mm[(num_steps - i) % num_steps] for i in range(num_steps)]
    return [d / 1000.0 for d in samples_ccw_mm]

def scan_and_localize():
    scan_m = run_map_scan()
    state = localizer.update_from_scan(scan_m)
    print(f"cell={state.cell}, heading={state.heading_deg:.1f}, confidence={state.confidence:.4f}")
    return state


2026-05-07 15:18:08,301 | INFO     |:  | Number of observations per grid cell: 18
2026-05-07 15:18:08,302 | INFO     |:  | Precaching Views...
2026-05-07 15:18:09,050 | INFO     |:  | Precaching Time: 0.748 secs
2026-05-07 15:18:09,050 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-05-07 15:18:09,050 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


In [38]:
# Optional debug only: run a 360 degree scan and print the Lab11 belief estimate.
# The main preset-path run does not need this cell.
# localizer.set_uniform_prior()
# state = scan_and_localize()


## Execute Preset Path

This section turns each preset segment into one relative turn and one timed straight drive. Start with the first-segment test before running the full path.


In [39]:
DMP_DELTA_SIGN = -1.0  # map headings are CCW; current DMP yaw convention is usually clockwise-positive.
LOG_PATH = Path("logs/lab12_preset_path_run.jsonl")

def append_run_log(record):
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

def drive_distance_ft(distance_ft,
                      ms_per_ft=MS_PER_FT,
                      base_pwm=BASE_PWM,
                      heading_kp=1.2,
                      front_stop_mm=FRONT_STOP_MM):
    duration_ms = int(round(distance_ft * ms_per_ft))
    distance_mm = distance_ft * 304.8
    return drive_cell(distance_mm=distance_mm,
                      base_pwm=base_pwm,
                      duration_ms=duration_ms,
                      heading_kp=heading_kp,
                      front_stop_mm=front_stop_mm)

def run_preset_path(path=PRESET_PATH_FT,
                    start_heading_deg=START_HEADING_DEG,
                    ms_per_ft=MS_PER_FT,
                    base_pwm=BASE_PWM,
                    heading_kp=1.2,
                    front_stop_mm=FRONT_STOP_MM,
                    turn_timeout_ms=5000,
                    turn_kp=0.8,
                    turn_ki=0.001,
                    turn_kd=0.2,
                    segment_start=0,
                    segment_stop=None,
                    skip_small_turn_deg=1.0,
                    settle_s=0.2,
                    collect_nav_log=False):
    mission_started_s = time.time()
    segments = preset_segments(path, start_heading_deg=start_heading_deg, ms_per_ft=ms_per_ft)
    if segment_stop is None:
        segment_stop = len(segments)
    selected_segments = segments[segment_start:segment_stop]

    print("Preset path:", " -> ".join(str(p) for p in path))
    print(f"Running segments {segment_start + 1} through {segment_stop}")

    append_run_log({
        "event": "mission_started",
        "mode": "preset_open_loop",
        "path": path,
        "start_heading_deg": start_heading_deg,
        "ms_per_ft": ms_per_ft,
        "base_pwm": base_pwm,
        "front_stop_mm": front_stop_mm,
        "wall_inflation_ft": WALL_INFLATION_FT,
        "time_s": mission_started_s,
    })

    for run_idx, seg in enumerate(selected_segments, start=segment_start + 1):
        cmd_delta = DMP_DELTA_SIGN * seg["turn_delta_deg"]
        print(
            f"\nSegment {run_idx}: {seg['start']} -> {seg['goal']} | "
            f"turn={seg['turn_delta_deg']:.1f} deg, cmd={cmd_delta:.1f} deg, "
            f"drive={seg['distance_ft']:.3f} ft, duration={seg['drive_duration_ms']} ms"
        )

        step_record = {
            "event": "segment",
            **seg,
            "run_segment": run_idx,
            "turn_cmd_deg": cmd_delta,
        }

        if abs(cmd_delta) <= skip_small_turn_deg:
            turn_status = "TURN_SKIPPED_SMALL_DELTA"
            turn_log = []
            print("  turn:", turn_status)
        else:
            turn_status = send_turn(cmd_delta,
                                    timeout_ms=turn_timeout_ms,
                                    kp=turn_kp,
                                    ki=turn_ki,
                                    kd=turn_kd)
            turn_log = request_nav_log() if collect_nav_log else []
            print("  turn:", turn_status)

        step_record["turn_status"] = turn_status
        step_record["turn_log"] = turn_log
        if not (turn_status.startswith("TURN_DONE") or turn_status == "TURN_SKIPPED_SMALL_DELTA"):
            append_run_log(step_record)
            print("Stopping because turn did not finish cleanly.")
            return

        drive_status = drive_distance_ft(seg["distance_ft"],
                                        ms_per_ft=ms_per_ft,
                                        base_pwm=base_pwm,
                                        heading_kp=heading_kp,
                                        front_stop_mm=front_stop_mm)
        drive_log = request_nav_log() if collect_nav_log else []
        print("  drive:", drive_status)

        step_record["drive_status"] = drive_status
        step_record["drive_log"] = drive_log
        step_record["tof_safety_stop"] = drive_status.startswith("DRIVE_STOPPED_TOF")
        append_run_log(step_record)

        if not drive_status.startswith("DRIVE_DONE"):
            print("Stopping because drive did not finish cleanly.")
            return

        ble.sleep(settle_s)

    append_run_log({"event": "mission_done", "path": path, "segments_run": [s["segment"] for s in selected_segments]})
    print("Preset path done.")
    print("Run log:", LOG_PATH)


In [67]:
stop_nav()

NAV_STOPPED


In [73]:
# Full run:
run_preset_path(
    base_pwm=46,
    ms_per_ft=750,
    front_stop_mm=10,
    turn_timeout_ms=8000,
    turn_kp=0.8,
    turn_ki=0,
    turn_kd=0.5,
)

Preset path: (-4, -3) -> (-2, -1) -> (1, -1) -> (2, -3) -> (5, -3) -> (5, -2) -> (5, 3) -> (0, 3) -> (0, 0)
Running segments 1 through 8

Segment 1: (-4, -3) -> (-2, -1) | turn=0.0 deg, cmd=-0.0 deg, drive=2.828 ft, duration=2121 ms
  turn: TURN_SKIPPED_SMALL_DELTA
DRIVE_STARTED,862.104,46,2121,-58.834,10
DRIVE_DONE,2130,-61.755,0.000
  drive: DRIVE_DONE,2130,-61.755,0.000

Segment 2: (-2, -1) -> (1, -1) | turn=-45.0 deg, cmd=45.0 deg, drive=3.000 ft, duration=2250 ms
TURN_STARTED,45.000,-61.796,-16.796,8000
TURN_DONE,-18.872,2.075
  turn: TURN_DONE,-18.872,2.075
DRIVE_STARTED,914.400,46,2250,-20.172,10
DRIVE_DONE,2253,-21.506,0.000
  drive: DRIVE_DONE,2253,-21.506,0.000

Segment 3: (1, -1) -> (2, -3) | turn=-63.4 deg, cmd=63.4 deg, drive=2.236 ft, duration=1677 ms
TURN_STARTED,63.434,-21.016,42.418,8000
TURN_DONE,39.853,2.565
  turn: TURN_DONE,39.853,2.565
DRIVE_STARTED,681.553,46,1677,40.193,10
DRIVE_DONE,1683,38.937,0.000
  drive: DRIVE_DONE,1683,38.937,0.000

Segment 4: (2, -3) -> 

In [ ]:
run_preset_path(segment_start=0, segment_stop=1, base_pwm=45, ms_per_ft=800, front_stop_mm=250)

In [ ]:
run_preset_path(segment_start=1, segment_stop=2, base_pwm=45, ms_per_ft=800, front_stop_mm=250)

In [ ]:
run_preset_path(segment_start=2, segment_stop=3, base_pwm=45, ms_per_ft=900, front_stop_mm=250)

In [ ]:
run_preset_path(segment_start=3, segment_stop=4, base_pwm=45, ms_per_ft=900, front_stop_mm=250)

In [ ]:
run_preset_path(segment_start=4, segment_stop=5, base_pwm=45, ms_per_ft=900, front_stop_mm=250)

In [ ]:
run_preset_path(segment_start=5, segment_stop=6, base_pwm=45, ms_per_ft=900, front_stop_mm=250)

In [48]:
send_turn(45, timeout_ms=5000, kp=0.8, ki=0, kd=0.5)

TURN_STARTED,45.000,-46.111,-1.111,5000
TURN_DONE,-1.200,0.089


'TURN_DONE,-1.200,0.089'

## Cleanup

Run these commands only after the robot has stopped or if you need to disconnect from BLE.


In [31]:
stop_nav()
ble.stop_notify(ble.uuid["RX_STRING"])
ble.disconnect()
